In [19]:
import re
import pandas as pd
import matplotlib.pyplot as plt
import math

In [24]:
FILE = '../data/raw_data/users.csv'

df = pd.read_csv(FILE)
users = df['user.index'].unique()
n_users = len(users)
cols = df.columns

print(cols.to_list())



['user.index', 'intake.survey.utime', 'intake.survey.tz', 'intake.survey.gmtoff', 'first.notif.utime', 'first.steps.utime', 'exit.survey.utime', 'exit.survey.tz', 'exit.survey.gmtoff', 'last.notif.utime', 'last.steps.utime', 'travel.start', 'travel.end', 'dropout.date', 'totaldays', 'own.phone', 'age', 'gender', 'marital', 'marother', 'ethnicity', 'ethother', 'household.size', 'children', 'childhome', 'occupation', 'education', 'vacation', 'screentime', 'sms', 'messaging', 'call', 'email', 'calendar', 'web', 'social', 'other.mobile', 'fitapp', 'fitapp.names', 'fittracker', 'fittracker.names', 'phonecomfort', 'compcomfort', 'office.shops', 'office.pleasant', 'office.sidewalk', 'consc.detail', 'consc.prepared', 'consc.carryplans', 'consc.startwork', 'consc.wastetime', 'consc.duties', 'consc.makeplans', 'consc', 'stairs.intake', 'walk.intake', 'parkfar.intake', 'workoutbreaks.intake', 'stand.intake', 'stairs.exit', 'walk.exit', 'parkfar.exit', 'workoutbreaks.exit', 'stand.exit', 'selfeff.

In [34]:
df.isnull().sum()

user.index              0
intake.survey.utime     0
intake.survey.tz        0
intake.survey.gmtoff    0
first.notif.utime       0
                       ..
vigact.metmins.exit     2
modact.metmins.exit     2
walk.metmins.exit       3
metmins.exit            3
ipaq.hepa.exit          3
Length: 117, dtype: int64

In [31]:
intake_cols = []
ext_cols = []
for col in cols:
    if 'intake' in col:
        intake_cols.append(col)
    if 'exit' in col:
        ext_cols.append(col)

print(intake_cols)
print(ext_cols)

['intake.survey.utime', 'intake.survey.tz', 'intake.survey.gmtoff', 'stairs.intake', 'walk.intake', 'parkfar.intake', 'workoutbreaks.intake', 'stand.intake', 'selfeff.tired.intake', 'selfeff.badmood.intake', 'selfeff.notime.intake', 'selfeff.vaca.intake', 'selfeff.precip.intake', 'selfeff.intake', 'vigact.days.intake', 'modact.days.intake', 'walk10.days.intake', 'sit.time.intake', 'vigact.time.intake', 'modact.time.intake', 'walk.time.intake', 'vigact.metmins.intake', 'modact.metmins.intake', 'walk.metmins.intake', 'metmins.intake', 'ipaq.hepa.intake']
['exit.survey.utime', 'exit.survey.tz', 'exit.survey.gmtoff', 'stairs.exit', 'walk.exit', 'parkfar.exit', 'workoutbreaks.exit', 'stand.exit', 'selfeff.tired.exit', 'selfeff.badmood.exit', 'selfeff.notime.exit', 'selfeff.vaca.exit', 'selfeff.precip.exit', 'selfeff.exit', 'vigact.days.exit', 'modact.days.exit', 'walk10.days.exit', 'sit.time.exit', 'vigact.time.exit', 'modact.time.exit', 'walk.time.exit', 'vigact.metmins.exit', 'modact.metm

In [37]:
intake_cols = ['stairs.intake','walk.intake', 'parkfar.intake', 'workoutbreaks.intake', 'stand.intake', 'selfeff.tired.intake', 'selfeff.badmood.intake', 'selfeff.notime.intake', 'selfeff.vaca.intake', 'selfeff.precip.intake', 'selfeff.intake', 'ipaq.hepa.intake']

# 找出所有 intake 列对应的变量名
stems = [c.replace('.intake', '') for c in intake_cols]

for s in stems:
    if s + '.exit' in df.columns:
        compare = df[['user.index', s + '.intake', s + '.exit']].copy()
        compare[s + '.diff'] = compare[s + '.exit'] - compare[s + '.intake']
        print(f"\n=== {s} ===")
        print(compare)






=== stairs ===
    user.index  stairs.intake  stairs.exit  stairs.diff
0            1              4          3.0         -1.0
1            2              3          4.0          1.0
2            3              4          4.0          0.0
3            4              5          5.0          0.0
4            5              2          2.0          0.0
5            6              4          4.0          0.0
6            7              3          3.0          0.0
7            8              3          3.0          0.0
8            9              3          4.0          1.0
9           10              4          4.0          0.0
10          11              2          4.0          2.0
11          12              4          5.0          1.0
12          13              4          3.0         -1.0
13          14              5          5.0          0.0
14          15              4          5.0          1.0
15          16              4          4.0          0.0
16          17              3   

In [40]:
cols = ['consc.carryplans', 'consc.startwork', 'consc.wastetime', 'consc.duties', 'consc.makeplans']

print(df[cols])

    consc.carryplans  consc.startwork  consc.wastetime  consc.duties  \
0                  5                1                1             2   
1                  4                2                3             3   
2                  4                2                2             2   
3                  5                1                1             1   
4                  4                2                4             1   
5                  5                2                1             3   
6                  4                2                1             1   
7                  4                1                1             1   
8                  4                3                4             2   
9                  4                2                3             1   
10                 3                1                2             2   
11                 4                4                4             2   
12                 4                3                4          

In [45]:
cols = ['age', 'gender', 'occupation']
print(df[cols])

    age  gender                                         occupation
0    48  female                                program coordinator
1    28    male                                            student
2    20  female                                    student, intern
3    21    male                      engineering intern, lifegaurd
4    33  female                                            student
5    20    male                                           engineer
6    41  female                                       statistician
7    24    male                                            student
8    22    male                                            student
9    20  female                                            student
10   19  female                                    research intern
11   25    male                                       grad student
12   22  female                                            student
13   54  female                                      clerk, de